<a href="https://colab.research.google.com/github/AaronParraMerino/detector-contradicciones-ia/blob/main/contradicciones_entrenar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install datasets torch

  Using cached https://github.com/explosion/spacy-models/releases/download/es_core_news_sm-3.8.0/es_core_news_sm-3.8.0-py3-none-any.whl (12.9 MB)
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import torch

# Check if a GPU is available
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"GPU está disponible! Nombre del dispositivo: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("GPU no encontrado. Usando la CPU en su lugar.")

GPU is available! Device name: Tesla T4


In [ ]:
from datasets import load_dataset
#Este no es nu modelo preentrenado ni nada, solo una base de datos de español.
dataset = load_dataset("facebook/xnli", "es")

def format_labels(example):
    label_map = {0: 0, 1: 0, 2: 1}
    example["binary_label"] = label_map.get(example["label"], 0)
    return example

dataset = dataset.map(format_labels)
train_data = dataset["validation"]
test_data = dataset["test"]

README.md:   0%|          | 0.00/20.8k [00:00<?, ?B/s]

es/train-00000-of-00001.parquet:   0%|          | 0.00/53.2M [00:00<?, ?B/s]

es/test-00000-of-00001.parquet:   0%|          | 0.00/342k [00:00<?, ?B/s]

es/validation-00000-of-00001.parquet:   0%|          | 0.00/173k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/392702 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5010 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2490 [00:00<?, ? examples/s]

Map:   0%|          | 0/392702 [00:00<?, ? examples/s]

Map:   0%|          | 0/5010 [00:00<?, ? examples/s]

Map:   0%|          | 0/2490 [00:00<?, ? examples/s]

In [ ]:
import re
import torch
from collections import Counter
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset

# 1. Regex Tokenizador
def tokenize(text):
    text = str(text).lower()
    # Mantener solo letras, números y espacios (quita la puntuación pero mantiene ñ/á)
    text = re.sub(r'[^\w\s]', '', text)
    return text.split()

# 2. Build the Vocabulary
print("Contruyendo el vocabulario desde 0...")
vocab = Counter()
for example in train_data:
    vocab.update(tokenize(example['premise']))
    vocab.update(tokenize(example['hypothesis']))

word2idx = {"<PAD>": 0, "<UNK>": 1}
for word, count in vocab.items():
    if count >= 2: # Ignorar errores tipográficos que aparecen una vez
        word2idx[word] = len(word2idx)

print(f"Vocabulary complete! Size: {len(word2idx)} words.")

Building vocabulary from scratch...
Vocabulary complete! Size: 5490 words.


In [ ]:
class NLIDataset(Dataset):
    def __init__(self, data, word2idx):
        self.data = data
        self.word2idx = word2idx

    def __len__(self):
        return len(self.data)

    def text_to_indices(self, text):
        tokens = tokenize(text)
        return [self.word2idx.get(token, 1) for token in tokens]

    def __getitem__(self, idx):
        item = self.data[idx]
        premise = torch.tensor(self.text_to_indices(item['premise']))
        hypothesis = torch.tensor(self.text_to_indices(item['hypothesis']))
        label = torch.tensor(item['binary_label'], dtype=torch.float32)
        return premise, hypothesis, label

def collate_fn(batch):
    premises, hypotheses, labels = zip(*batch)
    premises_padded = pad_sequence(premises, batch_first=True, padding_value=0)
    hypotheses_padded = pad_sequence(hypotheses, batch_first=True, padding_value=0)
    labels = torch.stack(labels)
    return premises_padded, hypotheses_padded, labels

train_loader = DataLoader(NLIDataset(train_data, word2idx), batch_size=32, collate_fn=collate_fn, shuffle=True)
test_loader = DataLoader(NLIDataset(test_data, word2idx), batch_size=32, collate_fn=collate_fn, shuffle=False)

print("Iniciando los cargadores de información para comenzar a entrenar al modelo!")

DataLoaders ready for training!


In [ ]:
import torch
import torch.nn as nn

class SiameseLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, dropout=0.2):
        super(SiameseLSTM, self).__init__()

        # 1. Embeddings de palabras compartidos: convierten IDs de palabras en vectores densos entrenables
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)

        # 2. LSTM compartida: usamos una LSTM bidireccional (BiLSTM) para obtener un mejor contexto
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True, bidirectional=True)

        # 3. Red clasificadora densa
        # Como la LSTM es bidireccional, el tamaño de su salida es hidden_dim * 2.
        # Concatenamos 4 características diferentes (u, v, |u-v|, u*v), por lo que multiplicamos por 4.
        linear_input_dim = (hidden_dim * 2) * 4

        self.fc = nn.Sequential(
            nn.Linear(linear_input_dim, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, output_dim)
        )

    def forward_once(self, x):
        # Esta función solo procesa una oración
        embedded = self.embedding(x)

        # El LSTM retorna la secuencia completa y los estados finales ocultos
        _, (hidden, _) = self.lstm(embedded)

        # Agarra el estado oculto de ambos tanto de adelante como atrás LSTM direcciones
        final_hidden = torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)
        return final_hidden

    def forward(self, premise, hypothesis):
        # 1. Pasa ambas oraciones a través de la misma red neuronal
        u = self.forward_once(premise)
        v = self.forward_once(hypothesis)

        # 2. Combina el vector de la oración para capturar su relación
        abs_diff = torch.abs(u - v)
        mul = u * v
        combined = torch.cat((u, v, abs_diff, mul), dim=1)

        # 3. Pasa las funcionalidades combinadas a través de capas densas
        out = self.fc(combined)
        return out

# Instanciamos el modelo
# We set output_dim=1 because this is a binary classification problem (Contradiction vs. Not)
# Ponemos set output_dim=1 porque esta es una clasificación binaria, para mantenerlo sencillo.
vocab_size = len(word2idx)
embedding_dim = 100
hidden_dim = 128

model = SiameseLSTM(vocab_size, embedding_dim, hidden_dim, output_dim=1)

# Movemos el modelo al GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

print(f"Modelo creado y movido a: {device}")
print(model)

Model built and moved to: cuda
SiameseLSTM(
  (embedding): Embedding(5490, 100, padding_idx=0)
  (lstm): LSTM(100, 128, batch_first=True, bidirectional=True)
  (fc): Sequential(
    (0): Linear(in_features=1024, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=256, out_features=1, bias=True)
  )
)


In [ ]:
import torch.optim as optim

# 1. Configurar el optimizador y la función de pérdida
optimizer = optim.Adam(model.parameters(), lr=0.001)

# BCEWithLogitsLoss combina internamente una función Sigmoid
# y la entropía cruzada binaria (Binary Cross Entropy)
criterion = nn.BCEWithLogitsLoss()

# Definir una función auxiliar para calcular la precisión (accuracy)
def calculate_accuracy(preds, labels):
    # Aplicar Sigmoid para convertir las salidas en valores entre 0 y 1
    # y luego redondear a 0 o 1
    rounded_preds = torch.round(torch.sigmoid(preds))

    # Comparar predicciones con etiquetas reales
    correct = (rounded_preds == labels).float()

    # Calcular la precisión como porcentaje de aciertos
    acc = correct.sum() / len(correct)

    return acc

# Número de épocas de entrenamiento
EPOCHS = 5

# Mensaje de inicio del entrenamiento
print("Iniciando entrenamiento...")

# Bucle principal de entrenamiento
for epoch in range(EPOCHS):

    # Colocar el modelo en modo entrenamiento
    model.train()

    # Variables para acumular pérdida y precisión de la época
    epoch_loss = 0
    epoch_acc = 0

    # Recorrer cada lote (batch) de datos
    for batch_idx, (premises, hypotheses, labels) in enumerate(train_loader):

        # 2. Mover los datos a la GPU o dispositivo seleccionado
        premises = premises.to(device)
        hypotheses = hypotheses.to(device)

        # Ajustar la forma de las etiquetas para que coincida
        # con la salida del modelo
        labels = labels.unsqueeze(1).to(device)

        # 3. Reiniciar los gradientes calculados anteriormente
        optimizer.zero_grad()

        # 4. Propagación hacia adelante (Forward Pass)
        # Obtener predicciones del modelo
        predictions = model(premises, hypotheses)

        # 5. Calcular la pérdida y la precisión
        loss = criterion(predictions, labels)
        acc = calculate_accuracy(predictions, labels)

        # 6. Propagación hacia atrás (Backward Pass)
        # Calcular gradientes mediante backpropagation
        loss.backward()

        # 7. Actualizar los pesos de la red neuronal
        optimizer.step()

        # Acumular métricas de la época
        epoch_loss += loss.item()
        epoch_acc += acc.item()

        # Mostrar progreso cada 50 lotes
        if batch_idx % 50 == 0:
            print(
                f"Época: {epoch+1}/{EPOCHS} | "
                f"Lote: {batch_idx}/{len(train_loader)} | "
                f"Pérdida del lote: {loss.item():.4f} | "
                f"Precisión del lote: {acc.item():.4f}"
            )

    # Calcular métricas promedio de la época
    avg_loss = epoch_loss / len(train_loader)
    avg_acc = epoch_acc / len(train_loader)

    # Mostrar resumen de la época
    print(f"--- Fin de la época {epoch+1} ---")
    print(
        f"Pérdida promedio de la época: {avg_loss:.4f} | "
        f"Precisión promedio de la época: {avg_acc:.4f}\n"
    )

# Mensaje final
print("¡Entrenamiento completado!")

Starting Training...
Epoch: 1/5 | Batch: 0/78 | Batch Loss: 0.7031 | Batch Acc: 0.3125
Epoch: 1/5 | Batch: 50/78 | Batch Loss: 0.5436 | Batch Acc: 0.7812
--- End of Epoch 1 ---
Average Epoch Loss: 0.6339 | Average Epoch Accuracy: 0.6627

Epoch: 2/5 | Batch: 0/78 | Batch Loss: 0.6057 | Batch Acc: 0.6250
Epoch: 2/5 | Batch: 50/78 | Batch Loss: 0.6196 | Batch Acc: 0.5625
--- End of Epoch 2 ---
Average Epoch Loss: 0.5755 | Average Epoch Accuracy: 0.7130

Epoch: 3/5 | Batch: 0/78 | Batch Loss: 0.5572 | Batch Acc: 0.7188
Epoch: 3/5 | Batch: 50/78 | Batch Loss: 0.5388 | Batch Acc: 0.6562
--- End of Epoch 3 ---
Average Epoch Loss: 0.4953 | Average Epoch Accuracy: 0.7558

Epoch: 4/5 | Batch: 0/78 | Batch Loss: 0.3578 | Batch Acc: 0.9062
Epoch: 4/5 | Batch: 50/78 | Batch Loss: 0.4756 | Batch Acc: 0.8125
--- End of Epoch 4 ---
Average Epoch Loss: 0.3824 | Average Epoch Accuracy: 0.8235

Epoch: 5/5 | Batch: 0/78 | Batch Loss: 0.3471 | Batch Acc: 0.8750
Epoch: 5/5 | Batch: 50/78 | Batch Loss: 0.442

In [ ]:
# 1. Crear el DataLoader para el conjunto de prueba
test_loader = DataLoader(
    NLIDataset(test_data, word2idx),
    batch_size=32,
    collate_fn=collate_fn,
    shuffle=False
)

# 2. Bucle de evaluación
# Colocar el modelo en modo evaluación
# (desactiva capas como Dropout)
model.eval()

# Variables para acumular pérdida y precisión
test_loss = 0
test_acc = 0

# Mensaje de inicio de evaluación
print("Evaluando en el conjunto de prueba...")

# torch.no_grad() indica a PyTorch que no calcule gradientes,
# ahorrando memoria y tiempo de cómputo
with torch.no_grad():

    # Recorrer todos los lotes del conjunto de prueba
    for premises, hypotheses, labels in test_loader:

        # Mover los datos al dispositivo (CPU o GPU)
        premises = premises.to(device)
        hypotheses = hypotheses.to(device)

        # Ajustar la forma de las etiquetas para coincidir
        # con la salida del modelo
        labels = labels.unsqueeze(1).to(device)

        # Obtener las predicciones del modelo
        predictions = model(premises, hypotheses)

        # Calcular la pérdida
        loss = criterion(predictions, labels)

        # Calcular la precisión
        acc = calculate_accuracy(predictions, labels)

        # Acumular métricas
        test_loss += loss.item()
        test_acc += acc.item()

# Calcular métricas promedio del conjunto de prueba
avg_test_loss = test_loss / len(test_loader)
avg_test_acc = test_acc / len(test_loader)

# Mostrar resultados finales
print(f"Pérdida final en prueba: {avg_test_loss:.4f}")
print(f"Precisión final en prueba: {avg_test_acc:.4f}")

Evaluating on Test Set...
Final Test Loss: 0.7523
Final Test Accuracy: 0.7064


In [ ]:
def predict_contradiction(premise, hypothesis, model, word2idx, device):

    # Colocar el modelo en modo evaluación
    model.eval()

    # 1. Tokenizar los textos y convertirlos a índices
    premise_tokens = tokenize(premise)
    hypo_tokens = tokenize(hypothesis)

    # Convertir cada palabra a su índice correspondiente
    # Si una palabra no existe en el vocabulario, usar <UNK> (índice 1)
    premise_indices = [word2idx.get(t, 1) for t in premise_tokens]
    hypo_indices = [word2idx.get(t, 1) for t in hypo_tokens]

    # 2. Convertir las listas de índices en tensores
    # unsqueeze(0) agrega la dimensión del lote (batch)
    premise_tensor = torch.tensor(premise_indices).unsqueeze(0).to(device)
    hypo_tensor = torch.tensor(hypo_indices).unsqueeze(0).to(device)

    # 3. Realizar la predicción
    with torch.no_grad():

        # Obtener la salida del modelo
        output = model(premise_tensor, hypo_tensor)

        # Convertir la salida a una probabilidad entre 0 y 1
        probability = torch.sigmoid(output).item()

        # Determinar la clase según el umbral de 0.5
        prediction = (
            "Contradicción"
            if probability > 0.5
            else "No contradicción"
        )

    # Mostrar resultados
    print(f"Premisa:     {premise}")
    print(f"Hipótesis:   {hypothesis}")
    print(
        f"Predicción:  {prediction} "
        f"(Confianza: {max(probability, 1 - probability):.2%})\n"
    )

    print("-" * 50)


# --- Probar oraciones personalizadas ---

# Ejemplo 1: Contradicción clara
predict_contradiction(
    "El gato está durmiendo",
    "El gato está roncando",
    model,
    word2idx,
    device
)

# Ejemplo 2: Implicación (No contradicción)
predict_contradiction(
    "El electricista instaló los cables en la casa nueva.",
    "Un trabajador completó la instalación eléctrica.",
    model,
    word2idx,
    device
)

# Ejemplo 3: Prueba personalizada
predict_contradiction(
    "La aplicación es para buscar trabajo informal.",
    "La aplicación solo ofrece empleos de oficina a tiempo completo.",
    model,
    word2idx,
    device
)

Premise:    El gato está durmiendo
Hypothesis: El gato está roncando
Prediction: Contradiction (Confidence: 52.79%)

--------------------------------------------------
Premise:    El electricista instaló los cables en la casa nueva.
Hypothesis: Un trabajador completó la instalación eléctrica.
Prediction: Not Contradiction (Confidence: 97.83%)

--------------------------------------------------
Premise:    La aplicación es para buscar trabajo informal.
Hypothesis: La aplicación solo ofrece empleos de oficina a tiempo completo.
Prediction: Not Contradiction (Confidence: 60.15%)

--------------------------------------------------


In [ ]:
import torch
import json
from google.colab import files

# 1. Guardar el modelo que hemos entrenado
model_path = 'siamese_lstm_model.pth'
torch.save(model.state_dict(), model_path)

# 2. Guardar el vocabulario
vocab_path = 'word2idx.json'
with open(vocab_path, 'w', encoding='utf-8') as f:
    json.dump(word2idx, f, ensure_ascii=False, indent=4)

# 3. Descargar
files.download(model_path)
files.download(vocab_path)

Model and Vocabulary saved to Colab environment.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>